# STSM Evaluation of Saved Models — Threshold from Normal Training Data

No training. For each run and window, the saved best model is loaded and evaluated with **one change**:
the anomaly threshold is the `pc`-th percentile of the reconstruction errors of the **normal (label-0)
training pairs**, instead of all training pairs.

Everything else is as in training: same windows, same data split, same scaler (fitted on normal training
pairs), same seismic criterion and seismic-ratio calculation.

Evaluate both the STSM model and the baseline LSTM-AE this way, so that they are compared with the same
threshold rule. The results are saved in the same format as the training results, under the name
`<model>-NormalThr` (e.g. `LSTM-TA-NormalThr`, `LSTM-AE-NormalThr`), so
`DEMETER_STSM-Results-Analysis.ipynb` can analyse them directly.

## Setup (PC or Google Colab)

In [ ]:
import sys
import subprocess
from pathlib import Path

LOCAL_PROJECT = Path(r"D:\GIT\Attention_Model-Demeter-Seismic-Study")
DRIVE_PROJECT = Path("/content/drive/MyDrive/[FuSe][STSM] Megha Babu/Attention model/Attention_Model-Demeter-Seismic-Study")
REPO_URL = "https://github.com/Meghababu99/Attention_Model-Demeter-Seismic-Study.git"
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)], check=True)
else:
    PROJECT = LOCAL_PROJECT

CODE_DIR = PROJECT / "Clean-Code"
sys.path.insert(0, str(CODE_DIR))
print("Project:", PROJECT)

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from lstm import SeismicCriteria, generate_windows, load_eq_catalog, prepare_window
from lstm.pipeline import process_split, run_seismic_analysis, plot_error_distributions
from lstm.STSM import RecurrentAutoencoder, model_name

## Settings

- `SOURCES`: for each model to evaluate, the folder with its saved best models and their file name.
  Files are searched in all subfolders, so any layout works.
- The baseline LSTM-AE models are the thesis models (`best_model_Hp_tw48-30dBG_{run}_w{i}.pth`, 20 windows).
  Set their folder; remove the entry to skip them.
- Runs with a missing model file for any window are skipped.

In [ ]:
# Data
DATA_DIR = PROJECT / "Data"
BG_WINDOW_DIR = DATA_DIR / "Bg_window_data"
LABEL_DIR = DATA_DIR / "Label_data"
EQ_PATH = DATA_DIR / "EQ.csv"

# Rolling windows (as in training)
train_months, val_months, stride = 12, 3, 3
WINDOW_START, WINDOW_END = "2005-01-01 00:00:00", "2011-01-02 00:00:00"
min_data_points = 17

# Anomaly threshold (percentile) and seismic criterion
pc = 98   # percentile of the NORMAL training errors
tw = 48   # time window (hours)
sw = 22   # spatial width (degrees)

# Models to evaluate: name -> (folder with the saved best models, file name)
SOURCES = {
    "LSTM-TA": (PROJECT / "Result Files" / "STSM-Results" / "LSTM-TA",
                "best_model_Hp_tw{tw}-30dBG_{run}-LSTM-TA_w{i}.pth"),
    "LSTM-AE": (PROJECT / "Models-Trained" / f"SW{sw}_TW{tw}",          # baseline (thesis) models
                "best_model_Hp_tw{tw}-30dBG_{run}_w{i}.pth"),
}
RUNS = list("ABCDE") + [f"In-{c}" for c in "ABCDE"]

# Outputs: Result Files/STSM-Results/<name>-NormalThr/
RESULTS_ROOT = PROJECT / "Result Files" / "STSM-Results"
RESUME = True   # skip windows already in the result CSV

def checkpoint(name, run, i):
    """Saved best model of a model, run and window (None if missing)."""
    folder, pattern = SOURCES[name]
    return next(folder.rglob(pattern.format(tw=tw, run=run, i=i)), None) if folder.is_dir() else None

## Model loading and evaluation with the normal-data threshold

`load_model` rebuilds the autoencoder from the checkpoint itself (cell type, bidirectional, attention,
hidden size, layers, latent size), so runs trained with different settings are loaded correctly.

In [ ]:
def load_model(path):
    """RecurrentAutoencoder with the architecture stored in the checkpoint."""
    state = torch.load(path, map_location="cpu")
    w = {k.replace(".lstm.", ".rnn."): v for k, v in state.items()}
    hh = w["encoder.rnn.weight_hh_l0"]
    hidden = hh.shape[1]
    model = RecurrentAutoencoder(
        input_size=w["encoder.rnn.weight_ih_l0"].shape[1],
        hidden_size=hidden,
        num_layers=sum(k.startswith("encoder.rnn.weight_ih_l") and not k.endswith("_reverse") for k in w),
        latent_dim=w["encoder.encoder_fc.weight"].shape[0],
        cell={4: "LSTM", 3: "GRU"}[hh.shape[0] // hidden],
        bidirectional=any(k.endswith("_reverse") for k in w),
        attention=("feature_temporal" if "encoder.feature_attention.weight" in w
                   else "temporal" if "encoder.temporal_attention.weight" in w else None),
    )
    model.load_state_dict(w)
    return model


def evaluate_window_normal_threshold(i, w, model, splits, datasets, scaled, eq, seismic_criteria, pc, m, output_dir):
    """
    As lstm.pipeline.evaluate_window, but the threshold is the pc-th percentile of the errors of the
    normal (label-0) training pairs. It is applied to all train and validation pairs.
    """
    normal = process_split(model, DataLoader(scaled["Train_normal"], batch_size=1), threshold_percentile=pc)
    threshold = normal["threshold_agg"]
    train = process_split(model, DataLoader(scaled["Train"], batch_size=1), threshold_agg=threshold)
    val = process_split(model, DataLoader(scaled["Val"], batch_size=1), threshold_agg=threshold)
    plot_error_distributions(i, m, train, val, pc, output_dir)

    rows = []
    for split, res in (("Val", val), ("Train", train)):
        sa = run_seismic_analysis(splits[split], datasets[split], eq, seismic_criteria, res["anomalies_agg"])
        rows.append({
            "window": i,
            "split": split,
            **w,
            "anomalies": len(res["anomalies_agg"]),
            "agg_value": sa["value"],
            "agg_error": sa["error"],
            "agg_total_eq": sa["total_eq"],
            "seismic_indices": sa["seismic_indices"],
            "threshold_agg": threshold,
            "threshold_source": "normal_train",
        })
    return rows

## Earthquake catalogue, rolling windows and available models

In [ ]:
eq = load_eq_catalog(EQ_PATH)
seismic_criteria = SeismicCriteria(spatial_width=sw, time_window_hours=tw)
windows = generate_windows(WINDOW_START, WINDOW_END, train_months, val_months, stride)

runs = {name: [run for run in RUNS if all(checkpoint(name, run, i) for i in range(len(windows)))] for name in SOURCES}

print("Windows:", len(windows))
for name, found in runs.items():
    print(f"{name}: runs with all models {found} | skipped {[run for run in RUNS if run not in found]}")

## Evaluation loop

In [ ]:
for name, run in [(name, run) for name in SOURCES for run in runs[name]]:
    m = f"tw{tw}-30dBG_{run}-{name}-NormalThr"
    output_dir = RESULTS_ROOT / f"{name}-NormalThr" / m / "outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    results_csv = output_dir / f"result_Hp_{m}.csv"

    done = pd.read_csv(results_csv) if RESUME and results_csv.exists() else pd.DataFrame(columns=["window"])
    results = done.to_dict("records")
    print(f"\n{'#' * 90}\nRun {run}: {m}\n{'#' * 90}")

    for i, w in enumerate(windows):
        if i in set(done["window"]):
            continue
        print(f"Window {i}: Val {w['val_start'].date()} -> {w['val_end'].date()}")
        splits, datasets, scaled = prepare_window(
            BG_WINDOW_DIR / f"Background_data-window_{i}.pkl",
            {s: LABEL_DIR / f"summary_df_{s.lower()}_30D-{sw}SW-tw{tw}_w{i}.csv" for s in ("Train", "Val")},
            w,
            min_data_points,
        )
        model = load_model(checkpoint(name, run, i))
        results += evaluate_window_normal_threshold(i, w, model, splits, datasets, scaled, eq, seismic_criteria,
                                                    pc, m, output_dir)
        pd.DataFrame(results).to_csv(results_csv, index=False)  # saved after every window

print("\nResults in:", [str(RESULTS_ROOT / f"{name}-NormalThr") for name in SOURCES])

## Next step

Open `DEMETER_STSM-Results-Analysis.ipynb` and set

```python
MODEL_NAME = "LSTM-TA-NormalThr"
BASELINE_NAME = "LSTM-AE-NormalThr"
```

to get the Z-factor, average ratio and the comparison with the baseline, both with the normal-data threshold.